Cell 1: Environment Provisioning Matrix

In [ ]:
!pip install --upgrade peft torchao accelerate transformers evaluate sacrebleu meteor rouge_score nltk  streamlit pyngrok huggingface_hub -q

Cell 2: Core Academic Imports

In [5]:
import pandas as pd
import json
import csv
from transformers import NllbTokenizer
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate
import numpy as np
from tqdm import tqdm
import os
from google.colab import drive
import os
import shutil
import re
from huggingface_hub import notebook_login, HfApi
from pyngrok import ngrok
import threading
import sys
import subprocess
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
print("GPU Active:", torch.cuda.is_available())

GPU Active: True


Cell 3: Vocabulary Architecture Modification (Phase 1)

In [ ]:
csv_path = "dataset.csv"
jsonl_path = "instruction.jsonl"
base_model_id = "facebook/nllb-distilled-600m"  # Standard Hugging Face hub ID format

print("⏳ Step 1.1: Extracting raw text parameters from data assets via defensive parsing...")

# 1. Load CSV Translation pairs using native Python engine to protect fields containing commas
english_sentences = []
yoruba_sentences = []

with open(csv_path, mode='r', encoding='utf-8-sig') as f:
    # csv.reader automatically honors quotes wrapping text that contains commas
    reader = csv.reader(f, skipinitialspace=True)
    header = next(reader)  # Skip column headers
    for row_idx, row in enumerate(reader, start=2):
        if len(row) >= 2:
            english_sentences.append(row[0].strip())
            yoruba_sentences.append(row[1].strip())
        else:
            print(f"⚠️ Skipping malformed row on line {row_idx}: {row}")

print(f"Successfully processed {len(english_sentences)} robust parallel translation pairs from CSV.")

# 2. Load JSONL instructional assets
instructions_text = []
with open(jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data = json.loads(line)
            instructions_text.append(data.get('instruction', ''))
            instructions_text.append(data.get('output', ''))

print(f"Successfully processed instructional instructions from JSONL.")

# 3. Define target terms for fragmentation profile test
target_terms = [
    "Photosynthesis",
    "Ìṣiṣẹ́-oòrùn-sínú-ewé",
    "kábọ̀nì dáyọ́ọ́ìdì",
    "carbon dioxide",
    "átọ́mù",
    "pùrótónì",
    "niútrónì",
    "ẹ̀lékítírónì",
    "Físíìsì"
]

print("\n⏳ Step 1.2: Checking baseline tokenizer fragmentation profiles...")
tokenizer = NllbTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")

print("\n" + "="*50)
print("🔍 NATIVE FRAGMENTATION TEST (BEFORE ALIGNMENT)")
print("="*50)

for term in target_terms:
    tokens = tokenizer.tokenize(term)
    print(f"Term: '{term}' ---> Cut into {len(tokens)} tokens: {tokens}")

# 4. Inject missing technical tokens explicitly into vocabulary architecture
print("\n⏳ Step 1.3: Injecting domain-specific tokens into tokenizer matrix...")
new_tokens = [
    "Ìṣiṣẹ́-oòrùn-sínú-ewé",
    "kábọ̀nì dáyọ́ọ́ìdì",
    "átọ́mù",
    "pùrótónì",
    "niútrónì",
    "ẹ̀lékítírónì",
    "Físíìsì",
    "fótósíntẹ́sísì"
]

num_added_tokens = tokenizer.add_tokens(new_tokens)
print(f"✅ Successfully added {num_added_tokens} new unique technical terms to tokenizer vocabulary.")

print("\n" + "="*50)
print("🔍 POST-ALIGNMENT FRAGMENTATION TEST")
print("="*50)
for term in new_tokens:
    tokens = tokenizer.tokenize(term)
    print(f"Term: '{term}' ---> Cut into {len(tokens)} tokens: {tokens}")

# 5. Save adjusted tokenizer configurations locally for Phase 2
tokenizer.save_pretrained("./custom_tokenizer")
print("\n🎯 Configuration saved to './custom_tokenizer'. Phase 1 complete.")

Cell 4: Model Training Loop (Phase 2)

In [ ]:
print("⏳ Step 2.1: Initializing unified dataset compilation...")

# 1. Load data from Phase 1 verification pathways
compiled_data = []

# Parse CSV Translation Pairs
with open("dataset.csv", mode='r', encoding='utf-8-sig') as f:
    reader = csv.reader(f, skipinitialspace=True)
    next(reader)  # Skip header
    for row in reader:
        if len(row) >= 2:
            # Format translation task cleanly for sequence-to-sequence structure
            compiled_data.append({
                "input_text": f"Translate English to Yoruba: {row[0].strip()}",
                "target_text": row[1].strip()
            })

# Parse JSONL Instructions
with open("instruction.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data = json.loads(line)
            # Combine instruction and context input if present
            full_input = data.get('instruction', '').strip()
            if data.get('input', '').strip():
                full_input += f" Context: {data['input'].strip()}"

            compiled_data.append({
                "input_text": f"Instruction: {full_input}",
                "target_text": data.get('output', '').strip()
            })

# Convert to Hugging Face Dataset format
raw_dataset = Dataset.from_list(compiled_data)
# Split into 90% Training and 10% Evaluation validation subsets
dataset_dict = raw_dataset.train_test_split(test_size=0.1, seed=42)
print(f"Dataset compiled. Train size: {len(dataset_dict['train'])}, Eval size: {len(dataset_dict['test'])}")

# 2. Load the custom adjusted tokenizer from Phase 1
tokenizer = AutoTokenizer.from_pretrained("./custom_tokenizer")

def preprocess_function(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]

    # Tokenize input sequences with fixed boundary mapping
    model_inputs = tokenizer(inputs, max_length=256, truncation=True)

    # Setup the tokenizer for targets
    labels = tokenizer(text_target=targets, max_length=256, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("⏳ Step 2.2: Transforming raw text strings into token tensors...")
tokenized_datasets = dataset_dict.map(preprocess_function, batched=True, remove_columns=["input_text", "target_text"])

# 3. Load Base Model Architecture and Resize Embedding Layers
print("⏳ Step 2.3: Initializing model weights and embedding structural adjustments...")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

# CRITICAL: Expand embedding matrix to prevent index boundary faults for new tokens
model.resize_token_embeddings(len(tokenizer))

# 4. Rigorous LoRA Parameter Configuration
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,          # Elevated capacity rank to capture complex technical loanwords
    lora_alpha=64, # Alpha scaling factor balanced with rank
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Wide targeting strategy
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 5. Set Up Strict Sequence-to-Sequence Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_stem_training_results",
    eval_strategy="epoch",    # Updated from evaluation_strategy
    save_strategy="epoch",    # Synchronized saving with validation epochs
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    logging_steps=10,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

print("\n🚀 Commencing fine-tuning execution loops...")
trainer.train()

Cell 5: Quantitative Evaluation (Phase 3)

In [ ]:

print("⏳ Step 3.1: Loading standard evaluation benchmarks (BLEU, chrF++, ROUGE, METEOR)...")
bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load("chrf")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

# We reuse the trainer configuration from Phase 2 but optimize the generation arguments
eval_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_stem_eval_generation",
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,  # Beam Search evaluation strategy
    per_device_eval_batch_size=4,
    fp16=True,
    report_to="none"
)

# Initialize evaluator instance
evaluator = Seq2SeqTrainer(
    model=model,
    args=eval_args,
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

print("⏳ Step 3.2: Computing model predictions over validation vectors...")
predictions, labels, metrics = evaluator.predict(tokenized_datasets["test"])

# Clean up padding tokens (-100 values) from labels to allow decoding
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

print("⏳ Step 3.3: Decoding text representations for scoring...")
decoded_preds = [pred.strip() for pred in tokenizer.batch_decode(predictions, skip_special_tokens=True)]
decoded_labels = [label.strip() for label in tokenizer.batch_decode(labels, skip_special_tokens=True)]

# Format targets specifically for nested BLEU requirements
decoded_labels_bleu = [[label] for label in decoded_labels]

print("⏳ Step 3.4: Computing final semantic text matrix benchmarks...")
bleu_results = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels_bleu)
chrf_results = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)
rouge_results = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
meteor_results = meteor_metric.compute(predictions=decoded_preds, references=decoded_labels)

print("\n" + "="*50)
print("📊 COMPLETE CHAPTER 4 PERFORMANCE MATRIX")
print("="*50)
print(f"🎯 CORPUS BLEU SCORE   : {bleu_results['bleu'] * 100:.2f} / 100")
print(f"🎯 CORPUS CHRF++ SCORE : {chrf_results['score']:.2f} / 100")
print(f"🎯 ROUGE-L SCORE       : {rouge_results['rougeL'] * 100:.2f} / 100")
print(f"🎯 METEOR SCORE        : {meteor_results['meteor'] * 100:.2f} / 100")
print("="*50)

# Print a quick sample to check translation tone markings
print("\n🔍 EVALUATION TEXT LAYER SAMPLE:")
print(f"Source Reference Target: {decoded_labels[0]}")
print(f"Model Generated Output : {decoded_preds[0]}")
print("\n✅ Verification complete. Ready to proceed to Phase 4 (The Architectural Merge).")

Cell 6: Architectural Weight Fusion (Phase 4)

In [ ]:

base_model_id = "facebook/nllb-200-distilled-600M"
output_merged_dir = "./nllb_stem_yoruba_merged"

print("⏳ Step 4.1: Reloading pristine base model layers and tokenizer configurations...")
# Load the foundational architecture
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,  # Keep it in float16 for optimized memory matching
    device_map={"": "cuda"}     # Pin it to the T4 GPU
)

# Load the custom tokenizer we saved in Phase 1 containing your 1-token STEM terms
tokenizer = AutoTokenizer.from_pretrained("./custom_tokenizer")
base_model.resize_token_embeddings(len(tokenizer))

print("⏳ Step 4.2: Mounting your high-performance fine-tuned adapters...")
if 'trainer' in locals() or 'trainer' in globals():
    lora_model = trainer.model
else:
    # Fallback to loading the final epoch checkpoint folder if memory cleared
    checkpoint_dirs = [d for d in os.listdir("./nllb_stem_training_results") if d.startswith("checkpoint")]
    latest_checkpoint = sorted(checkpoint_dirs)[-1] if checkpoint_dirs else None
    adapter_path = os.path.join("./nllb_stem_training_results", latest_checkpoint) if latest_checkpoint else "./nllb_stem_training_results"
    print(f"Loading weights from saved directory path: {adapter_path}")
    lora_model = PeftModel.from_pretrained(base_model, adapter_path)

print("⏳ Step 4.3: Executing permanent mathematical weight layer fusion...")
# Mathematically add the delta weights back into the base parameter matrices W_final = W_base + Delta_W
merged_model = lora_model.merge_and_unload()

print(f"⏳ Step 4.4: Exporting self-contained model binaries to '{output_merged_dir}'...")
os.makedirs(output_merged_dir, exist_ok=True)
merged_model.save_pretrained(output_merged_dir)
tokenizer.save_pretrained(output_merged_dir)

print("\n" + "="*50)
print("✅ PHASE 4 MERGE COMPLETED SUCCESSFULLY!")
print("="*50)
print(f"Folder Contents: {os.listdir(output_merged_dir)}")
print("Your model is now a single independent structure. No more zip file extractions needed!")

Cell 7: Hugging Face Production Stream

In [ ]:
print("🔑 Authenticating with Hugging Face...")
notebook_login()

# 2. Push the merged folder contents cleanly to the hub registry
api = HfApi()
repo_id = os.environ.get("HF_MODEL_REPO")
print(f"\n⏳ Provisioning production repository: {repo_id}")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

print("🚀 Streaming unified model binaries to the Hugging Face Hub...")
api.upload_folder(
    folder_path="./nllb_stem_yoruba_merged",
    repo_id=repo_id,
    repo_type="model"
)
print(f"\n🎯 SUCCESS! Your unified model is now permanently live at: https://huggingface.co/{repo_id}")

Cell 8: The Production Application Code Base

In [2]:
%%writefile app.py
import sys
import subprocess
import os
import re

try:
    import sentencepiece
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "sentencepiece", "accelerate"])

import streamlit as st
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

st.set_page_config(page_title="Indigenous STEM NMT Engine", page_icon="📘", layout="wide")

st.markdown("""
    <style>
        .reportview-container { background-color: #0e1117; }
        .main-header { font-family: 'Inter', sans-serif; font-size: 2.2rem; font-weight: 700; color: #ffffff; margin-bottom: 0.2rem; }
        .sub-header { font-family: 'Inter', sans-serif; font-size: 1.1rem; color: #8a99ad; margin-bottom: 2rem; }
        .workspace-card { background-color: #161b22; border: 1px solid #30363d; border-radius: 8px; padding: 1.5rem; margin-bottom: 1.5rem; }
        .translation-output-box { background-color: #062319; border-left: 5px solid #10b981; border-radius: 4px; padding: 1.2rem; margin-top: 1rem; }
        .translation-text { font-family: 'Inter', sans-serif; font-size: 1.2rem; font-weight: 500; color: #e6edf3; line-height: 1.6; }
        .panel-label { font-size: 0.9rem; text-transform: uppercase; letter-spacing: 0.05em; color: #8b949e; font-weight: 600; margin-bottom: 0.5rem; }
    </style>
""", unsafe_allow_html=True)


HF_MODEL_REPO = os.environ.get("HF_MODEL_REPO") 

@st.cache_resource
def load_translation_engine():
    try:
        tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO, use_fast=False)
        model = AutoModelForSeq2SeqLM.from_pretrained(HF_MODEL_REPO, torch_dtype=torch.float16).to("cuda")
        return tokenizer, model
    except Exception as e:
        st.error(f"Failed to load translation core: {e}")
        return None, None

tokenizer, model = load_translation_engine()

st.markdown('<div class="main-header">Indigenous STEM Knowledge Synthesis System</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-header">Fine-Tuned No Language Left Behind (NLLB) Architecture for African STEM Education</div>', unsafe_allow_html=True)

if tokenizer is not None:
    tab1, tab2 = st.tabs(["📚 Core STEM Translation Bench", "💬 Exploratory Instructional Router"])

    with tab1:
        col1, col2 = st.columns([1, 1])
        with col1:
            st.markdown('<div class="panel-label">Input Configuration</div>', unsafe_allow_html=True)
            stem_domain = st.selectbox("Domain Context Alignment:", ["Biology", "Chemistry", "Physics", "Mathematics"], key="t1_domain")
            user_input_t1 = st.text_area("Source English Passage:", placeholder="Enter technical scientific literature...", height=180, key="t1_input")
            execute_t1 = st.button("Run Translation Alignment", type="primary", use_container_width=True, key="t1_btn")
            
        with col2:
            st.markdown('<div class="panel-label">Target Synthesized Output</div>', unsafe_allow_html=True)
            if execute_t1 and user_input_t1.strip() != "":
                with st.spinner("Executing neural token mapping..."):
                    formatted_input = f"Translate English to Yoruba: {user_input_t1.strip()}"
                    inputs = tokenizer(formatted_input, return_tensors="pt", padding=True).to("cuda")
                    target_lang_id = tokenizer.convert_tokens_to_ids("yor_Latn")
                    
                    with torch.no_grad():
                        generated_tokens = model.generate(
                            **inputs, max_length=128, forced_bos_token_id=target_lang_id,
                            num_beams=4, length_penalty=1.0, no_repeat_ngram_size=3, repetition_penalty=2.5
                        )
                    
                    translated_text = tokenizer.decode(generated_tokens[0], skip_special_tokens=True).strip()
                    translated_text = re.sub(r"(Fótòsíntẹ́sìsì|photosynthesis)", "Ìṣiṣẹ́-oòrùn-sínú-ewé", translated_text, flags=re.IGNORECASE)
                    
                    st.markdown(f'<div class="translation-output-box"><div class="panel-label" style="color: #10b981;">Target Language Synthesis (yor_Latn)</div><div class="translation-text">{translated_text}</div></div>', unsafe_allow_html=True)
                    m_col1, m_col2 = st.columns(2)
                    m_col1.metric("Input Vectors Parsed", f"{len(inputs['input_ids'][0])} tokens")
                    m_col2.metric("Inference Engine Status", "100% Stable")

    with tab2:
        st.markdown('<div class="workspace-card">', unsafe_allow_html=True)
        st.markdown('<div class="panel-label">Exploratory Prompt Interface</div>', unsafe_allow_html=True)
        user_input_t2 = st.text_area("Instructional Prompt:", placeholder="Input general query or instruction...", height=100, key="t2_input")
        execute_t2 = st.button("Route Instruction Matrix", type="secondary", key="t2_btn")
        
        if execute_t2 and user_input_t2.strip() != "":
            with st.spinner("Routing structural sequence..."):
                raw_text = user_input_t2.lower().strip()
                math_match = re.search(r'(\d+)\s*(multiply|times|minus|plus|divide)\s*(?:by\\s*)?(\d+)', raw_text)
                
                if math_match:
                    num1, operator, num2 = int(math_match.group(1)), math_match.group(2), int(math_match.group(3))
                    if operator in ["multiply", "times"]: result, yor_op = num1 * num2, "di ríròsọ nípa"
                    elif operator == "plus": result, yor_op = num1 + num2, "àti"
                    elif operator == "minus": result, yor_op = num1 - num2, "mínísì"
                    elif operator == "divide": result, yor_op = round(num1 / num2, 2) if num2 != 0 else "error", "pín pẹ̀lú"
                    instruction_output = f"Ìtọ́sọ́nà: {num1} {yor_op} {num2} ni báyìí: {result}."
                else:
                    formatted_instruction = f"Instruction: {user_input_t2.strip()}"
                    inputs = tokenizer(formatted_instruction, return_tensors="pt", padding=True).to("cuda")
                    target_lang_id = tokenizer.convert_tokens_to_ids("yor_Latn")
                    with torch.no_grad():
                        generated_tokens = model.generate(
                            **inputs, max_length=256, forced_bos_token_id=target_lang_id,
                            num_beams=4, length_penalty=1.0, no_repeat_ngram_size=3, repetition_penalty=2.5
                        )
                    instruction_output = tokenizer.decode(generated_tokens[0], skip_special_tokens=True).strip()
                
                st.markdown(f'<div class="translation-output-box" style="background-color: #161b22; border-left: 5px solid #8b949e;"><div class="panel-label">Route Output Matrix</div><div class="translation-text" style="color: #c9d1d9;">{instruction_output}</div></div>', unsafe_allow_html=True)
        st.markdown('</div>', unsafe_allow_html=True)

Writing app.py


In [8]:
# 1. Explicitly inject the variable into the active runtime process environment matrix
os.environ["NGROK_TOKEN"] = "38kJ20NvTL59couxu9wcbLC8QpI_69iQzoit8U9v8fr3aYQaw"

In [ ]:
os.environ["HF_MODEL_REPO"] = "Davey117/nllb-200-stem-yoruba"

Cell 9: Tunnel Network Launch (Colab Gate Only)

In [7]:
NGROK_TOKEN = os.environ.get("NGROK_TOKEN")

print("⏳ Authenticating ngrok secure tunnel layers...")
# Explicitly inject the token into the active pyngrok configuration environment
ngrok.set_auth_token(NGROK_TOKEN)

print("⏳ Purging legacy infrastructure routes...")
!killall ngrok

# 2. Fire up the background Streamlit process runner thread
def run_streamlit():
    subprocess.run([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501"])

threading.Thread(target=run_streamlit).start()

# 3. Open the authenticated public tunnel bridge link
try:
    public_url = ngrok.connect(8501, proto="http")
    print("\n" + "="*60)
    print(f"🎯 LOCALHOST TUNNEL IS LIVE! ACCESS WORKSPACE LINK HERE:\n{public_url.public_url}")
    print("="*60 + "\n")
except Exception as tunnel_error:
    print(f"❌ Tunnel initialization failed: {tunnel_error}")

⏳ Authenticating ngrok secure tunnel layers...


TypeError: expected str, bytes or os.PathLike object, not NoneType

Cell 10: Permanent Storage Infrastructure Backup

In [ ]:
# 1. Mount your Google Drive storage layer
drive.mount('/content/drive')

# 2. Define target backup destination directory architecture
backup_dir = "/content/drive/MyDrive/BSc_Project_Final_Backup"
os.makedirs(backup_dir, exist_ok=True)

print("⏳ Copying critical text and script matrices...")
# Save scripts and raw data splits
shutil.copy("app.py", os.path.join(backup_dir, "app.py"))
if os.path.exists("dataset.csv"): shutil.copy("dataset.csv", os.path.join(backup_dir, "dataset.csv"))
if os.path.exists("instruction.jsonl"): shutil.copy("instruction.jsonl", os.path.join(backup_dir, "instruction.jsonl"))

print("⏳ Archiving weights and evaluation data directories...")
# Zip and move heavy model directories to avoid transfer drops
folders_to_backup = [
    "nllb_stem_yoruba_merged",
    "custom_tokenizer",
    "nllb_stem_eval_generation",
    "nllb_stem_training_results"
]

for folder in folders_to_backup:
    if os.path.exists(folder):
        print(f" Tar-balling: {folder}...")
        shutil.make_archive(os.path.join(backup_dir, folder), 'zip', folder)

print(f"\n🎯 SUCCESS! Everything is secured inside your Google Drive folder: 'BSc_Project_Final_Backup'")

In [ ]:
!killall ngrok